In [0]:
%pip install torch

# IMPORTS

In [0]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset
print(torch.__version__)
print(f"GPU disponible: {torch.cuda.is_available()}")

In [0]:
import requests
import mlflow
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import collect_list,struct
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from pyspark.ml.feature import MinMaxScaler, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array
from pyspark.sql.types import NumericType
from pyspark.sql.functions import col, isnan, when,lit
from pyspark.sql.types import NumericType
import datetime

In [0]:
spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.getOrCreate()

# Data load

In [0]:
prices = spark.read.format('delta').load('/Volumes/market-mood/features/feature_store')

In [0]:
prices.printSchema()

# Division

In [0]:
fs_pdf = prices.dropna().toPandas()#convertir a pandas para poder usarlo para entrenar
fs_pdf = fs_pdf.sort_values(["ticker", "date"]).reset_index(drop=True)
fs_pdf.info()
fs_pdf['date'] = pd.to_datetime(fs_pdf['date'])
FEATURE_COLS = ["close", "volume", "log_return", "rsi", "macd", 
                "bb_position", "volatility", "vol_ratio"]
TARGET_COL = "target"
SEQUENCE_LENGTH = 20

# Split temporal
train = fs_pdf[fs_pdf["date"] < "2023-01-01"]
val   = fs_pdf[(fs_pdf["date"] >= "2023-01-01") & (fs_pdf["date"] < "2024-01-01")]
test  = fs_pdf[fs_pdf["date"] >= "2024-01-01"]

print(f"Train: {len(train)} filas")
print(f"Val:   {len(val)} filas")
print(f"Test:  {len(test)} filas")

# Normalization

In [0]:
global_features = [ 'log_return',
 'rsi',
 'macd',
 'bb_position',
 'volatility',
 'vol_ratio']
ticker_features = ['close','volume']

In [0]:
scaler = StandardScaler()
for col in global_features:
    train[col] = scaler.fit_transform(train[[col]])
    test[col] = scaler.fit_transform(test[[col]])
    val[col] = scaler.fit_transform(val[[col]])



In [0]:
tickers = train['ticker'].unique()

In [0]:
scalers_by_ticker = {}
for ticker in tickers:
    scaler = StandardScaler().fit(train[train['ticker'] == ticker][['close','volume']])
    scalers_by_ticker[ticker] = scaler
for ticker in tickers:
    train.loc[train['ticker']==ticker,ticker_features] = scalers_by_ticker[ticker].transform(train[train['ticker'] == ticker][ticker_features])

In [0]:
scalers_by_ticker = {}
for ticker in tickers:
    scaler = StandardScaler().fit(test[test['ticker'] == ticker][['close','volume']])
    scalers_by_ticker[ticker] = scaler
for ticker in tickers:
    test.loc[test['ticker']==ticker,ticker_features] = scalers_by_ticker[ticker].transform(test[test['ticker'] == ticker][ticker_features])

In [0]:
scalers_by_ticker = {}
for ticker in tickers:
    scaler = StandardScaler().fit(val[val['ticker'] == ticker][['close','volume']])
    scalers_by_ticker[ticker] = scaler
for ticker in tickers:
    val.loc[val['ticker']==ticker,ticker_features] = scalers_by_ticker[ticker].transform(val[val['ticker'] == ticker][ticker_features])

## Encoding

In [0]:
ticker_dummies = pd.get_dummies(train["ticker"], prefix="ticker")
train = pd.concat([train, ticker_dummies], axis=1)
ticker_dummies = pd.get_dummies(test["ticker"], prefix="ticker")
test = pd.concat([test, ticker_dummies], axis=1)
ticker_dummies = pd.get_dummies(val["ticker"], prefix="ticker")
val = pd.concat([val, ticker_dummies], axis=1)


In [0]:
val.columns

In [0]:
ticker_cols = []
for col in train.columns:
    if col.startswith('ticker_'): 
        ticker_cols.append(col)


In [0]:
ticker_cols

In [0]:
train = spark.createDataFrame(train)
test = spark.createDataFrame(test)
val = spark.createDataFrame(val)

In [0]:
train.printSchema()

# Format data for entry

In [0]:
context_size = 20
w = Window.partitionBy('ticker').orderBy('date').rowsBetween(-(context_size-1),0)
train = train.withColumn('context',collect_list(struct('close','volume','log_return','rsi','macd','bb_position','volatility','vol_ratio','ticker_AAPL',
 'ticker_ADBE',
 'ticker_AMZN',
 'ticker_BAC',
 'ticker_BRK-B',
 'ticker_DIS',
 'ticker_GOOGL',
 'ticker_HD',
 'ticker_JNJ',
 'ticker_JPM',
 'ticker_MA',
 'ticker_META',
 'ticker_MSFT',
 'ticker_NFLX',
 'ticker_NVDA',
 'ticker_PG',
 'ticker_PYPL',
 'ticker_TSLA',
 'ticker_UNH',
 'ticker_V')).over(w))

test = test.withColumn('context',collect_list(struct('close','volume','log_return','rsi','macd','bb_position','volatility','vol_ratio','ticker_AAPL',
 'ticker_ADBE',
 'ticker_AMZN',
 'ticker_BAC',
 'ticker_BRK-B',
 'ticker_DIS',
 'ticker_GOOGL',
 'ticker_HD',
 'ticker_JNJ',
 'ticker_JPM',
 'ticker_MA',
 'ticker_META',
 'ticker_MSFT',
 'ticker_NFLX',
 'ticker_NVDA',
 'ticker_PG',
 'ticker_PYPL',
 'ticker_TSLA',
 'ticker_UNH',
 'ticker_V')).over(w))
val = val.withColumn('context',collect_list(struct('close','volume','log_return','rsi','macd','bb_position','volatility','vol_ratio','ticker_AAPL',
 'ticker_ADBE',
 'ticker_AMZN',
 'ticker_BAC',
 'ticker_BRK-B',
 'ticker_DIS',
 'ticker_GOOGL',
 'ticker_HD',
 'ticker_JNJ',
 'ticker_JPM',
 'ticker_MA',
 'ticker_META',
 'ticker_MSFT',
 'ticker_NFLX',
 'ticker_NVDA',
 'ticker_PG',
 'ticker_PYPL',
 'ticker_TSLA',
 'ticker_UNH',
 'ticker_V')).over(w))


In [0]:
train.printSchema()

In [0]:
train = train.dropna().toPandas()#convertir a pandas para poder usarlo para entrenar
test = test.dropna().toPandas()#convertir a pandas para poder usarlo para entrenar
val = val.dropna().toPandas()#convertir a pandas para poder usarlo para entrenar
val = val.sort_values(["ticker", "date"]).reset_index(drop=True)
test = test.sort_values(["ticker", "date"]).reset_index(drop=True)
train = train.sort_values(["ticker", "date"]).reset_index(drop=True)

In [0]:
print(f"Shape: {fs_pdf.shape}")
print(f"Tickers: {fs_pdf['ticker'].nunique()}")
print(f"Rango: {fs_pdf['date'].min()} → {fs_pdf['date'].max()}")

In [0]:

train['date'] = pd.to_datetime(train['date'])
test['date'] = pd.to_datetime(test['date'])
val['date'] = pd.to_datetime(val['date'])

In [0]:
def change_format(context_list):
    return np.array([[v for v in d.values()] for d in context_list])

# Convertir context a arrays (20, 8)
train["context"] = train["context"].map(change_format)
val["context"]   = val["context"].map(change_format)
test["context"]  = test["context"].map(change_format)

print(f"Train: {len(train)} filas")
print(f"Val:   {len(val)} filas")
print(f"Test:  {len(test)} filas")

In [0]:
train = train[train["context"].map(lambda x: x.shape[0]) == 20]
val   = val[val["context"].map(lambda x: x.shape[0]) == 20]
test  = test[test["context"].map(lambda x: x.shape[0]) == 20]

print(f"Train: {len(train)} filas")
print(f"Val:   {len(val)} filas")
print(f"Test:  {len(test)} filas")

# Training LSTM

## All variables no exogen

In [0]:
#20 del one hot encoding, más 8 de features numéricas

# rnn = torch.nn.LSTM(
#     input_size=28,
#     hidden_size=64,
#     num_layers=2,#más capas, mas overfitting
#     #dropout = 0.2,
#     barch_first = True,# por si le matriz de entrada está girada
# )

class MarketLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim,dropout=0.2):
        super(MarketLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.dropout = dropout
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, dropout=dropout,batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, h0=None, c0=None):
        if h0 is None or c0 is None:
            h0 = torch.zeros(self.layer_dim, x.size(#inicializar los arrays de memoria
                0), self.hidden_dim).to(x.device)
            c0 = torch.zeros(self.layer_dim, x.size(
                0), self.hidden_dim).to(x.device)

        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])  # Take last time step
        return out, hn, cn
lstm = MarketLSTM(28,64,2,1,0.3)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(lstm.parameters(),lr=0.001)

In [0]:
print(lstm)

In [0]:


class MarketDataset(Dataset):
    def __init__(self, df):
        self.df = df
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = torch.tensor(row["context"], dtype=torch.float32)
        y = torch.tensor(row["target"], dtype=torch.float32)
        return x,y


In [0]:
train_dataset = MarketDataset(train)
val_dataset   = MarketDataset(val)
test_dataset  = MarketDataset(test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)#dividir en batches los datos
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

In [0]:
train["target"].value_counts(normalize=True)

In [0]:
epochs = 100

with mlflow.start_run(run_name="lstm-technical-all-features"):
    mlflow.log_params({
        "input_dim": 28,
        "hidden_dim": 64,
        "num_layers": 2,
        "dropout": 0.3,
        "learning_rate": 0.001,
        "batch_size": 32,
        "epochs": epochs,
        "features": "all"
    })
    for epoch in range(epochs):
        train_loss = 0
        lstm.train()
        for x_batch,y_batch in train_loader:
            optimizer.zero_grad()#reiniciar optimizador para que optimice de acuerdo al batch y epoch actual
            outputs,_,_ = lstm(x_batch)
            outputs = outputs.squeeze(1)#aplastar para que sea un vector y no una matriz de una fila
            loss = criterion(outputs, y_batch)
            train_loss += loss.item()
            loss.backward()#calcular gradientes con backpropagation
            optimizer.step()#actualizar pesos

        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item()}')
        lstm.eval()#se modifican los modos del modelo para que no aprenda al insertar datos nuevos
        val_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                outputs,_,_ = lstm(x_batch)
                outputs = outputs.squeeze(1)#aplastar para que sea un vector y no una matriz de una fila
                loss = criterion(outputs, y_batch)
                prob = torch.sigmoid(outputs)
                pred = (prob > 0.5).float()
                correct += (pred == y_batch).sum().item()
                total += y_batch.size(0)
                val_loss += loss.item()

        mlflow.log_metrics({
            "train_loss": train_loss / len(train_loader),
            "val_loss": val_loss / len(val_loader),
            "val_accuracy": correct / total
        }, step=epoch)

        

In [0]:
# Baseline tonto: siempre predice 1
y_true = val["target"].values
baseline_accuracy = y_true.mean()
print(f"Accuracy si siempre predices 1: {baseline_accuracy:.4f}")
print(f"Accuracy si siempre predices 0: {1-baseline_accuracy:.4f}")

# Verificar que el DataLoader mantiene el orden correcto
first_batch_targets = next(iter(val_loader))[1]
print(f"Primeros targets del val_loader: {first_batch_targets[:10]}")
print(f"Primeros targets del DataFrame: {val['target'].values[:10]}")

In [0]:
# Guardar el modelo en MLflow Registry
mlflow.pytorch.log_model(
    lstm,
    artifact_path="market-lstm-technical",
    registered_model_name="market-mood-lstm-technical"
)

In [0]:
epochs = 100
h0, c0 = None, None
for epoch in range(epochs):
    lstm.train()
    optimizer.zero_grad()
    outputs,h0,c0 = lstm(train['context'],h0,c0)
    loss = criterion(outputs, train['target'])
    
    loss.backward()
    optimizer.step()

    h0,c0=h0.detach(),c0.detach()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')